# 07 · 控制生成：img2img、inpainting、LoRA

純文字生圖(text2img)像擲骰子——同一句 prompt 每次都不一樣。真正用在創作上,你需要**控制**。這課介紹三招業界最常用的控制手段。

> - **img2img**:給一張**起始圖** + prompt,讓模型在它的基礎上變化(保留構圖、換風格)。
> - **inpainting**:只重繪圖片的**局部**(用遮罩指定),其餘不動(去除路人、換背景)。
> - **LoRA**:用少量圖**微調**出一個風格/角色的小外掛,掛上去就能穩定生成特定風格。

## 1. 安裝

In [ ]:
%pip install -q -U diffusers transformers accelerate

## 2. img2img:在一張圖的基礎上變化

`strength` 控制變化幅度(0=幾乎不變,1=幾乎重畫)。sd-turbo 下 `steps×strength` 要 ≥ 1。

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image

pipe = AutoPipelineForImage2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16
).to("cuda")

init = load_image(
    "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
).resize((512, 512))
out = pipe("a fantasy landscape, vibrant oil painting", image=init,
           num_inference_steps=2, strength=0.6, guidance_scale=0.0).images[0]

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(init); ax[0].set_title("起始圖"); ax[0].axis("off")
ax[1].imshow(out); ax[1].set_title("img2img 後"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 3. inpainting:只重繪局部(概念)

inpainting 用一張**遮罩**(白=要重繪、黑=保留),搭配專門的 inpaint 模型。流程示意如下:

In [ ]:
# inpainting 流程示意(需 inpaint 專用權重,Colab 上可換成 sd 系列 inpaint 模型):
# from diffusers import AutoPipelineForInpainting
# pipe = AutoPipelineForInpainting.from_pretrained(
#     "stabilityai/stable-diffusion-2-inpainting", torch_dtype=torch.float16).to("cuda")
# result = pipe(prompt="a cat sitting on the bench",
#               image=init_image, mask_image=mask).images[0]
print("inpainting:用遮罩指定『只重畫這塊』,其餘像素原封不動——修圖、去路人、換背景的利器。")

## 4. LoRA:幫模型加一個風格外掛(概念)

LoRA 用少量圖訓練出一個**很小的權重補丁**(幾 MB),掛上基礎模型就能穩定生成特定角色/畫風,不用重訓整個幾 GB 的模型。這是社群分享風格的主流格式。

In [ ]:
# 掛載一個 LoRA 外掛(示意):
# pipe.load_lora_weights("some-author/some-style-lora")
# image = pipe("a portrait in <that-style>", num_inference_steps=1, guidance_scale=0.0).images[0]
print("LoRA = 風格/角色的小外掛(幾 MB),掛上基礎模型即可,不必重訓大模型。")
print("Civitai / Hugging Face 上有海量社群 LoRA 可下載。")

## 小結

- **img2img**:在起始圖上變化,`strength` 控制幅度——保構圖、換風格。
- **inpainting**:用遮罩只重繪局部,其餘不動——精準修圖。
- **LoRA**:少量圖訓練的小權重外掛,讓模型穩定產出特定風格/角色,免重訓大模型。
- 這三招把擲骰子般的生成,變成**可控的創作工具**。

下一課(壓軸):把這些組成一個**自己的圖像生成小工具**,並聊怎麼分享上線。